# EEG_22 — Trial Confidence Analysis: Firma EEG dei Trial Decodificabili

**Domanda**: i trial che il modello DHSLP (EEG_13b) decodifica correttamente hanno una firma EEG
diversa da quelli sbagliati? E quella firma dipende dal cluster strutturale (C0 fronto-motor vs C1 fronto-occipital)?

**Ipotesi**:
- C0: trial corretti → gamma più alta in aree fronto-centrali (F2, C2, FC2)
- C1: trial corretti → gamma più alta nel network F3–PO8

**Dati**: checkpoint EEG_13b (`models/eeg13b_200e/P{sid:03d}.pt`), sessione test (sessione 5) di ogni soggetto top.

**Pipeline**:
1. Carica modello EEG_13b per soggetto
2. Inference sulla sessione test → per-trial (y_true, y_pred, confidence, correct?)
3. Band power (5 bande) per ogni trial × elettrodo
4. Confronto corretto vs sbagliato — separato per cluster C0/C1
5. Mann-Whitney U per elettrodo × banda → topomaps effect size

## §1 — Setup

In [ ]:
import json, logging, re
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import signal
from scipy.stats import mannwhitneyu
from sklearn.metrics import balanced_accuracy_score
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s %(levelname)-8s %(message)s',
                    datefmt='%H:%M:%S')
log = logging.getLogger('eeg22')

project_root = next((p for p in [Path.cwd()] + list(Path.cwd().parents)
                     if (p / '.git').exists()), Path.cwd())
FIG_DIR  = project_root / 'figures'; FIG_DIR.mkdir(exist_ok=True)
CKPT_13B = project_root / 'models' / 'eeg13b_200e'
CKPT_DIR = project_root / 'models' / 'eeg22'; CKPT_DIR.mkdir(parents=True, exist_ok=True)

# === CONFIG identico a EEG_13b ===
N_CHANNELS   = 61
N_SAMPLES    = 384
N_CLASSES    = 4
CLUSTER_SCHEME = 'concr4'
K_WINDOWS    = 8
N_EDGES      = 16
D_MODEL      = 64
HIDDEN       = 128
N_LAYERS     = 2
DROPOUT      = 0.5
T_WIN        = N_SAMPLES // K_WINDOWS   # 48
FS           = 256   # Hz
DATA_METRIC  = 'abs_pcc'

# Cluster colori
CLUSTER_NAMES  = {0: 'Fronto-motor (C0)', 1: 'Fronto-occipital (C1)'}
CLUSTER_COLORS = {0: '#4A90E2', 1: '#FF8C42'}

# Bande spettrali
BANDS = {
    'delta': (1, 4),
    'theta': (4, 8),
    'alpha': (8, 13),
    'beta':  (13, 30),
    'gamma': (30, 50),
}

# === Label mapping ===
label2cluster = {int(k): int(v) for k, v in json.loads(
    (project_root/'configs'/'label_schemes'/'labelid2cluster_concr4.json').read_text()).items()}
_CLASS_NAMES = ['CONCR', 'AZIONE', 'STATO', 'ASTRATTO']

# === Indice hypergraph ===
HG_ROOT  = project_root / 'data' / f'hypergraphs_pruned_{DATA_METRIC}'
_PAT     = re.compile(r'^P(\d+)_S(\d+)$')
subj_sess = defaultdict(lambda: defaultdict(list))
for p in sorted(HG_ROOT.rglob('trial_*.pt')):
    m = _PAT.match(p.parent.name)
    if m:
        subj_sess[int(m.group(1))][int(m.group(2))].append(p)

device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps'  if torch.backends.mps.is_available() else 'cpu')
log.info(f'Device: {device}')
log.info(f'CKPT_13B: {CKPT_13B}  esiste={CKPT_13B.exists()}')
log.info(f'Soggetti trovati: {len(subj_sess)}')

## §2 — Architettura DHSLP + carica cluster labels

In [ ]:
# === DHSLP — identica a EEG_13b ===
class HGNNConv(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(in_dim, out_dim))
        self.bias   = nn.Parameter(torch.zeros(out_dim))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x, H):
        # x: (B, N, in_dim)  H: (B, N, E)
        Dv   = H.sum(dim=2, keepdim=True).clamp(min=1e-6)   # (B, N, 1)
        De   = H.sum(dim=1).clamp(min=1e-6).unsqueeze(2)    # (B, E, 1)
        Ht   = H.transpose(1, 2)                             # (B, E, N)
        x_norm = x / Dv                                      # (B, N, in_dim)
        step1  = torch.bmm(Ht, x_norm)                       # (B, E, in_dim)
        step2  = step1 / De                                   # (B, E, in_dim)
        agg    = torch.bmm(H, step2)                          # (B, N, in_dim)
        return agg @ self.weight + self.bias


class DHSLP(nn.Module):
    def __init__(self, n_nodes=N_CHANNELS, T_win=T_WIN, K=K_WINDOWS,
                 n_edges=N_EDGES, d_model=D_MODEL, hidden=HIDDEN,
                 n_classes=N_CLASSES, n_layers=N_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.K, self.T_win, self.d_model = K, T_win, d_model
        self.E       = nn.Parameter(torch.randn(n_edges, d_model) * 0.01)
        self.pos_enc = nn.Parameter(torch.randn(n_nodes, d_model) * 0.01)
        self.node_proj = nn.Sequential(
            nn.Linear(T_win, d_model),
            nn.LayerNorm(d_model),
            nn.ELU(),
        )
        dims = [d_model] + [hidden] * n_layers
        self.convs = nn.ModuleList([HGNNConv(dims[i], dims[i+1]) for i in range(n_layers)])
        self.bns   = nn.ModuleList([nn.BatchNorm1d(hidden) for _ in range(n_layers)])
        self.drop  = nn.Dropout(dropout)
        self.clf   = nn.Linear(hidden, n_classes)

    def build_dynamic_H(self, feat):
        scores = torch.matmul(feat, self.E.T) / (self.d_model ** 0.5)
        return torch.softmax(scores, dim=2)

    def forward(self, x):
        B, N, T = x.shape
        outs = []
        for k in range(self.K):
            x_k  = x[:, :, k*self.T_win:(k+1)*self.T_win]
            feat = self.node_proj(x_k) + self.pos_enc
            H_k  = self.build_dynamic_H(feat)
            out  = feat
            for conv, bn in zip(self.convs, self.bns):
                out = conv(out, H_k)
                out = bn(out.reshape(B*N, -1)).reshape(B, N, -1)
                out = F.relu(out)
                out = self.drop(out)
            outs.append(out.mean(dim=1))
        return self.clf(torch.stack(outs, dim=1).mean(dim=1))


def load_model(subj_id):
    """Carica checkpoint EEG_13b per soggetto. Ritorna modello in eval mode."""
    ckpt_path = CKPT_13B / f'P{subj_id:03d}.pt'
    if not ckpt_path.exists():
        log.warning(f'Checkpoint mancante: {ckpt_path}')
        return None
    model = DHSLP().to(device)
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ck['state_dict'])
    model.eval()
    log.info(f'P{subj_id:03d} caricato — val_bacc={ck["val_bacc"]:.4f}  test_bacc={ck["test_bacc"]:.4f}')
    return model


# === Cluster labels da EEG_16b ===
labels_path = project_root / 'configs' / 'eeg16b_cluster_labels.json'
if labels_path.exists():
    _cd = json.loads(labels_path.read_text())
    SUBJ_CLUSTER = {s: l for s, l in zip(_cd['subj_ids'], _cd['labels'])}
    log.info(f'Cluster labels caricati: C0={sum(v==0 for v in SUBJ_CLUSTER.values())}  C1={sum(v==1 for v in SUBJ_CLUSTER.values())}')
else:
    log.warning('eeg16b_cluster_labels.json non trovato — esegui §22 di EEG_16b')
    SUBJ_CLUSTER = {}

## §3 — Inference → dataframe per-trial

Per ogni soggetto top: carica il modello EEG_13b, esegui inference sulla **sessione test** (sessione con indice più alto),
raccogli per ogni trial: y_true, y_pred, confidenza (max softmax), corretto?

In [ ]:
# Soggetti da analizzare — top per bAcc EEG_13b, con checkpoint disponibile
# Priorità: soggetti di entrambi i cluster per il confronto C0 vs C1
ANALYZE_SUBJ = []  # auto-rilevato sotto

# Trova tutti i soggetti con checkpoint disponibile
available = sorted([int(p.stem[1:]) for p in CKPT_13B.glob('P*.pt')])
log.info(f'Checkpoint disponibili: {len(available)} soggetti')

# Prendi top-N con checkpoint, bilanciati per cluster
TOP_N = 10   # per cluster (se disponibili)
c0_subjs = [s for s in available if SUBJ_CLUSTER.get(s) == 0]
c1_subjs = [s for s in available if SUBJ_CLUSTER.get(s) == 1]
ANALYZE_SUBJ = c0_subjs[:TOP_N] + c1_subjs[:TOP_N]
log.info(f'Analisi su: C0={len(c0_subjs[:TOP_N])}  C1={len(c1_subjs[:TOP_N])}  TOT={len(ANALYZE_SUBJ)}')

In [ ]:
def band_power(x_np, fs=FS):
    """
    Calcola potenza per banda per ogni elettrodo.
    Input:  x_np (N_CHAN, T)
    Output: dict band -> array (N_CHAN,)
    """
    freqs, psd = signal.welch(x_np, fs=fs, nperseg=min(128, x_np.shape[1]),
                               axis=1)  # psd: (N_CHAN, n_freqs)
    result = {}
    for band, (lo, hi) in BANDS.items():
        mask = (freqs >= lo) & (freqs < hi)
        result[band] = psd[:, mask].mean(axis=1)   # (N_CHAN,)
    return result


def run_inference(subj_id, model):
    """
    Inference sulla sessione test (ultima disponibile).
    Ritorna lista di dict: {path, y_true, y_pred, confidence, correct, x_np, band_power}
    """
    sids = sorted(subj_sess[subj_id].keys())
    if len(sids) < 2:
        return []
    test_sess = sids[-1]
    paths = subj_sess[subj_id][test_sess]

    records = []
    with torch.no_grad():
        for p in paths:
            d = torch.load(p, weights_only=False)
            x = d['x'].float()
            y_word = int(d['y'].squeeze()) if isinstance(d['y'], torch.Tensor) else int(d['y'])
            y_true = label2cluster.get(y_word)
            if y_true is None:
                continue

            # Instance norm
            x_norm = (x - x.mean(dim=1, keepdim=True)) / (x.std(dim=1, keepdim=True) + 1e-6)
            x_in   = x_norm.unsqueeze(0).to(device)   # (1, 61, 384)

            logits = model(x_in)                       # (1, 4)
            probs  = F.softmax(logits, dim=1).squeeze().cpu().numpy()  # (4,)
            y_pred = int(probs.argmax())
            conf   = float(probs.max())

            # Band power sul segnale RAW (non normalizzato) per confronto fisico
            x_np  = x.numpy()   # (61, 384)
            bp    = band_power(x_np)

            records.append({
                'subj_id':   subj_id,
                'sess':      test_sess,
                'path':      str(p),
                'y_true':    y_true,
                'y_pred':    y_pred,
                'confidence': conf,
                'correct':   int(y_pred == y_true),
                'probs':     probs,
                'x_np':      x_np,
                'bp':        bp,
            })
    return records


# === Esegui inference su tutti i soggetti ===
ALL_RECORDS = []
MISSING_CKPT = []

for sid in tqdm(ANALYZE_SUBJ, desc='Inference soggetti'):
    model = load_model(sid)
    if model is None:
        MISSING_CKPT.append(sid)
        continue
    recs = run_inference(sid, model)
    ALL_RECORDS.extend(recs)
    bacc = balanced_accuracy_score([r['y_true'] for r in recs],
                                    [r['y_pred'] for r in recs])
    pct_correct = np.mean([r['correct'] for r in recs]) * 100
    cluster_name = CLUSTER_NAMES.get(SUBJ_CLUSTER.get(sid, -1), '?')
    log.info(f'P{sid:03d} [{cluster_name}] — {len(recs)} trial  bAcc={bacc:.4f}  corretto={pct_correct:.1f}%')
    del model  # libera memoria

df = pd.DataFrame([{k: v for k, v in r.items() if k not in ('x_np', 'bp', 'probs')}
                    for r in ALL_RECORDS])
df['cluster'] = df['subj_id'].map(SUBJ_CLUSTER)
print(f'\nTotale trial: {len(df)}')
print(df.groupby(['cluster', 'correct']).size().unstack(fill_value=0))
if MISSING_CKPT:
    print(f'\n⚠️ Checkpoint mancanti: {MISSING_CKPT}')

## §4 — Band power: corretto vs sbagliato per cluster

Per ogni cluster (C0/C1): confronta la potenza spettrale media
dei trial corretti vs incorretti per ogni elettrodo e banda.

Metrica: differenza relativa `(mean_correct - mean_wrong) / mean_all`

In [ ]:
def compute_cluster_bp_diff(records, cluster_id):
    """
    Per un cluster: ritorna
      bp_correct[band] = (N_correct, N_CHAN)
      bp_wrong[band]   = (N_wrong,   N_CHAN)
      diff[band]       = (N_CHAN,) differenza relativa media
    """
    recs_c = [r for r in records if SUBJ_CLUSTER.get(r['subj_id']) == cluster_id]
    correct = [r for r in recs_c if r['correct'] == 1]
    wrong   = [r for r in recs_c if r['correct'] == 0]

    log.info(f'Cluster {cluster_id}: corretto={len(correct)}  sbagliato={len(wrong)}')

    bp_ok  = {b: np.stack([r['bp'][b] for r in correct]) for b in BANDS} if correct else {}
    bp_no  = {b: np.stack([r['bp'][b] for r in wrong])   for b in BANDS} if wrong  else {}

    diff = {}
    pval = {}
    cohd = {}
    for band in BANDS:
        if not correct or not wrong:
            diff[band] = np.zeros(N_CHANNELS)
            pval[band] = np.ones(N_CHANNELS)
            cohd[band] = np.zeros(N_CHANNELS)
            continue
        m_ok = bp_ok[band].mean(axis=0)   # (N_CHAN,)
        m_no = bp_no[band].mean(axis=0)
        m_all = (m_ok * len(correct) + m_no * len(wrong)) / (len(correct) + len(wrong))
        diff[band] = (m_ok - m_no) / (m_all + 1e-12)

        # Mann-Whitney U per elettrodo
        pv = np.ones(N_CHANNELS)
        cd = np.zeros(N_CHANNELS)
        for ch in range(N_CHANNELS):
            stat, p = mannwhitneyu(bp_ok[band][:, ch], bp_no[band][:, ch],
                                   alternative='two-sided')
            pv[ch] = p
            # Cohen's d
            pooled_std = np.sqrt((bp_ok[band][:, ch].std()**2 +
                                  bp_no[band][:, ch].std()**2) / 2 + 1e-12)
            cd[ch] = (m_ok[ch] - m_no[ch]) / pooled_std
        pval[band] = pv
        cohd[band] = cd

    return bp_ok, bp_no, diff, pval, cohd


RESULTS = {}
for cl in [0, 1]:
    print(f'\n--- Cluster {cl} ({CLUSTER_NAMES[cl]}) ---')
    bp_ok, bp_no, diff, pval, cohd = compute_cluster_bp_diff(ALL_RECORDS, cl)
    RESULTS[cl] = {'bp_ok': bp_ok, 'bp_no': bp_no, 'diff': diff, 'pval': pval, 'cohd': cohd}

    # Top elettrodi per gamma
    if 'gamma' in diff:
        top_ch = np.argsort(np.abs(diff['gamma']))[::-1][:10]
        print(f'Top-10 elettrodi per |diff| gamma: canali {top_ch.tolist()}')
        sig_ch = np.where(pval['gamma'] < 0.05)[0]
        print(f'Elettrodi gamma p<0.05: {len(sig_ch)}/61 → {sig_ch.tolist()}')

## §5 — Topomap differenza per banda

Visualizza la differenza (corretto − sbagliato) / all per ogni banda e cluster.
Usa MNE per le topomaps.

In [ ]:
try:
    import mne
    # Carica nomi canali reali dal dataset (evita mismatch con standard_1020)
    _sample_pt = next(iter(next(iter(subj_sess.values())).values()))
    _d = torch.load(_sample_pt[0], weights_only=False)
    if 'ch_names' in _d:
        CH_NAMES = _d['ch_names']
    else:
        # Fallback: nomi BrainProducts 61ch standard del dataset
        CH_NAMES = ['Fp1','Fp2','F7','F3','Fz','F4','F8','FC5','FC1','FC2',
                    'FC6','T7','C3','Cz','C4','T8','TP9','CP5','CP1','CP2',
                    'CP6','TP10','P7','P3','Pz','P4','P8','PO9','O1','Oz',
                    'O2','PO10','AF7','AF3','AF4','AF8','F5','F1','F2','F6',
                    'FT9','FT7','FC3','FC4','FT8','FT10','C5','C1','C2','C6',
                    'TP7','CP3','CPz','CP4','TP8','P5','P1','P2','P6','PO7',
                    'PO3','POz','PO4','PO8'][:N_CHANNELS]
    _info = mne.create_info(ch_names=CH_NAMES, sfreq=FS, ch_types='eeg')
    _mont = mne.channels.make_standard_montage('standard_1020')
    _info.set_montage(_mont, match_case=False, on_missing='ignore')
    HAS_MNE = True
    log.info(f'MNE disponibile — topomap attive ({len(CH_NAMES)} canali)')
except Exception as e:
    HAS_MNE = False
    log.warning(f'MNE non disponibile: {e} — uso bar chart per elettrodo')


def plot_bp_diff_topomap(results, cluster_id, save_path=None):
    """Topomap differenza (corretto-sbagliato)/all per ogni banda."""
    diff = results[cluster_id]['diff']
    pval = results[cluster_id]['pval']
    cohd = results[cluster_id]['cohd']
    title = CLUSTER_NAMES[cluster_id]
    color = CLUSTER_COLORS[cluster_id]
    bands = list(BANDS.keys())

    fig, axes = plt.subplots(1, len(bands), figsize=(4 * len(bands), 4.5))
    fig.patch.set_facecolor('#1A1F2E')
    fig.suptitle(f'{title} — Δ band power (corretto − sbagliato) / all\n'
                 f'Rosso = corretto ha più potenza | Blu = sbagliato ha più potenza',
                 color='white', fontsize=12, fontweight='bold')

    vmax = max(np.abs(diff[b]).max() for b in bands if len(diff[b]) > 0)
    vmax = max(vmax, 0.01)  # evita vmax=0

    for ax, band in zip(axes, bands):
        ax.set_facecolor('#1A1F2E')
        data = diff[band]
        sig  = pval[band] < 0.05
        n_sig = sig.sum()

        if HAS_MNE:
            try:
                im, _ = mne.viz.plot_topomap(
                    data, _info, axes=ax, show=False,
                    cmap='RdBu_r', vlim=(-vmax, vmax),
                    sensors=True, contours=4
                )
                ax.set_title(f'{band.upper()}\n{n_sig}/61 p<.05',
                             color='white', fontsize=10)
            except Exception:
                # fallback bar chart
                ax.bar(range(len(data)), data,
                       color=[('#E63946' if d > 0 else '#4A90E2') for d in data],
                       alpha=0.8)
                ax.axhline(0, color='white', lw=0.5)
                ax.set_title(f'{band.upper()}\n{n_sig}/61 p<.05',
                             color='white', fontsize=10)
                ax.set_facecolor('#2A3142')
        else:
            ax.bar(range(len(data)), data,
                   color=[('#E63946' if d > 0 else '#4A90E2') for d in data],
                   alpha=0.8)
            ax.axhline(0, color='white', lw=0.5)
            ax.set_title(f'{band.upper()}\n{n_sig}/61 p<.05',
                         color='white', fontsize=10)
            ax.set_facecolor('#2A3142')
            ax.tick_params(colors='white', labelsize=7)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=130, bbox_inches='tight', facecolor='#1A1F2E')
        log.info(f'Salvato: {save_path}')
    plt.show()


for cl in [0, 1]:
    plot_bp_diff_topomap(RESULTS, cl,
                          save_path=FIG_DIR / f'eeg22_bp_diff_C{cl}.png')

## §6 — Gamma spotlight: C0 vs C1

Banda gamma è la più rilevante per imagined speech (Li et al. 2025).
Plot affiancato C0 vs C1: dove è più alta la potenza gamma nei trial corretti?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#1A1F2E')
fig.suptitle('Gamma (30–50 Hz): Δ potenza corretto − sbagliato per cluster\n'
             'Ipotesi: C0 → picco fronto-centrale | C1 → picco F3/PO8',
             color='white', fontsize=12, fontweight='bold')

for ax, cl in zip(axes, [0, 1]):
    data  = RESULTS[cl]['diff']['gamma']   # (61,)
    cohd  = RESULTS[cl]['cohd']['gamma']
    pval  = RESULTS[cl]['pval']['gamma']
    color = CLUSTER_COLORS[cl]
    sig   = pval < 0.05

    bars = ax.bar(range(N_CHANNELS), data,
                  color=['#E63946' if d > 0 else '#4A90E2' for d in data],
                  alpha=0.75, width=0.9)
    # Marca significativi con asterisco
    for ch in np.where(sig)[0]:
        y_pos = data[ch] + (0.005 if data[ch] >= 0 else -0.015)
        ax.text(ch, y_pos, '*', ha='center', va='bottom', color='white', fontsize=8)

    ax.axhline(0, color='white', lw=0.5)
    ax.set_facecolor('#2A3142')
    ax.set_title(f'{CLUSTER_NAMES[cl]}\n{sig.sum()}/61 elettrodi p<0.05',
                 color='white', fontsize=11)
    ax.set_xlabel('Canale EEG (0–60)', color='white')
    ax.set_ylabel('Δ relativo (corretto − sbagliato) / all', color='white')
    ax.tick_params(colors='white')

    # Top-5 Cohen's d
    top5 = np.argsort(np.abs(cohd))[::-1][:5]
    print(f'\nC{cl} — top-5 Cohen\'s d gamma:')
    for ch in top5:
        print(f'  ch={ch:2d}  d={cohd[ch]:+.3f}  p={pval[ch]:.4f}  sig={"★" if sig[ch] else ""}')

plt.tight_layout()
plt.savefig(FIG_DIR / 'eeg22_gamma_spotlight.png', dpi=130,
            bbox_inches='tight', facecolor='#1A1F2E')
plt.show()

## §7 — Confidenza per classe semantica

Quale classe semantica (CONCR/AZIONE/STATO/ASTRATTO) ha confidenza media più alta?
Collega con §2 di EEG_21 (per-class recall).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#1A1F2E')
fig.suptitle('Confidenza media per classe semantica — C0 vs C1',
             color='white', fontsize=12, fontweight='bold')

for ax, cl in zip(axes, [0, 1]):
    recs_c = [r for r in ALL_RECORDS if SUBJ_CLUSTER.get(r['subj_id']) == cl]
    if not recs_c:
        ax.set_visible(False)
        continue

    conf_by_class = defaultdict(list)
    acc_by_class  = defaultdict(list)
    for r in recs_c:
        conf_by_class[r['y_true']].append(r['confidence'])
        acc_by_class[r['y_true']].append(r['correct'])

    classes = list(range(N_CLASSES))
    means_conf = [np.mean(conf_by_class[c]) if conf_by_class[c] else 0 for c in classes]
    means_acc  = [np.mean(acc_by_class[c])  if acc_by_class[c]  else 0 for c in classes]
    stds_conf  = [np.std(conf_by_class[c])  if conf_by_class[c] else 0 for c in classes]

    x = np.arange(N_CLASSES)
    bars = ax.bar(x - 0.2, means_conf, 0.35, label='Confidenza media',
                  color='#FF8C42', alpha=0.85, yerr=stds_conf, capsize=4)
    bars2 = ax.bar(x + 0.2, means_acc, 0.35, label='Accuracy',
                   color='#52B788', alpha=0.85)
    ax.axhline(1/N_CLASSES, color='red', lw=1, ls='--', label=f'Chance ({1/N_CLASSES:.2f})')

    ax.set_facecolor('#2A3142')
    ax.set_xticks(x)
    ax.set_xticklabels(_CLASS_NAMES, color='white')
    ax.tick_params(colors='white')
    ax.set_title(f'{CLUSTER_NAMES[cl]}', color='white', fontsize=11)
    ax.set_ylabel('Valore medio', color='white')
    ax.legend(facecolor='#2A3142', labelcolor='white', fontsize=9)
    ax.set_ylim(0, max(0.6, max(means_conf) + 0.05))

    print(f'\nC{cl} per classe:')
    for c in classes:
        print(f'  {_CLASS_NAMES[c]:10s}  conf={means_conf[c]:.3f}±{stds_conf[c]:.3f}  acc={means_acc[c]:.3f}  N={len(conf_by_class[c])}')

plt.tight_layout()
plt.savefig(FIG_DIR / 'eeg22_class_confidence.png', dpi=130,
            bbox_inches='tight', facecolor='#1A1F2E')
plt.show()

## §8 — Connettività: corretto vs sbagliato

Confronta la matrice adj media dei trial corretti vs sbagliati per cluster.
Domanda: i trial correttamente decodificati hanno connettività più simile alla topografia del cluster?

In [ ]:
def compute_mean_adj(records, correct_flag):
    """Calcola adj media (pearson correlation) sui trial filtrati."""
    recs = [r for r in records if r['correct'] == correct_flag]
    if not recs:
        return np.zeros((N_CHANNELS, N_CHANNELS))
    adjs = []
    for r in recs:
        x = r['x_np']   # (61, 384)
        # abs PCC
        xn = x - x.mean(axis=1, keepdims=True)
        norms = np.linalg.norm(xn, axis=1, keepdims=True) + 1e-12
        xn = xn / norms
        adj = np.abs(xn @ xn.T)
        np.fill_diagonal(adj, 0)
        adjs.append(adj)
    return np.mean(adjs, axis=0)  # (61, 61)


fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.patch.set_facecolor('#1A1F2E')
fig.suptitle('Connettività media: corretto vs sbagliato per cluster',
             color='white', fontsize=12, fontweight='bold')

for row, cl in enumerate([0, 1]):
    recs_c = [r for r in ALL_RECORDS if SUBJ_CLUSTER.get(r['subj_id']) == cl]
    adj_ok = compute_mean_adj(recs_c, correct_flag=1)
    adj_no = compute_mean_adj(recs_c, correct_flag=0)
    adj_diff = adj_ok - adj_no

    vmax_adj = max(adj_ok.max(), adj_no.max())
    vmax_diff = np.abs(adj_diff).max()

    for col, (mat, title) in enumerate([
        (adj_ok,   f'C{cl} — Corretti\n(N={sum(1 for r in recs_c if r["correct"]==1)})'),
        (adj_no,   f'C{cl} — Sbagliati\n(N={sum(1 for r in recs_c if r["correct"]==0)})'),
        (adj_diff, f'C{cl} — Differenza\n(corretto − sbagliato)'),
    ]):
        ax = axes[row, col]
        ax.set_facecolor('#1A1F2E')
        cmap = 'hot' if col < 2 else 'RdBu_r'
        vmax = vmax_adj if col < 2 else vmax_diff
        vmin = 0 if col < 2 else -vmax
        im = ax.imshow(mat, cmap=cmap, vmin=vmin, vmax=vmax, aspect='auto')
        ax.set_title(title, color='white', fontsize=10)
        ax.tick_params(colors='white', labelsize=7)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig(FIG_DIR / 'eeg22_connectivity_diff.png', dpi=130,
            bbox_inches='tight', facecolor='#1A1F2E')
plt.show()

## §9 — Riepilogo e Conclusioni

In [ ]:
print('=' * 60)
print('EEG_22 — RIEPILOGO')
print('=' * 60)

for cl in [0, 1]:
    recs_c = [r for r in ALL_RECORDS if SUBJ_CLUSTER.get(r['subj_id']) == cl]
    n_ok = sum(1 for r in recs_c if r['correct'] == 1)
    n_no = sum(1 for r in recs_c if r['correct'] == 0)
    print(f'\n{CLUSTER_NAMES[cl]}:')
    print(f'  Trial totali={len(recs_c)}  corretti={n_ok}  sbagliati={n_no}')

    for band in BANDS:
        n_sig = (RESULTS[cl]['pval'][band] < 0.05).sum()
        max_d = np.abs(RESULTS[cl]['cohd'][band]).max()
        print(f'  {band:6s}: {n_sig:2d}/61 sig  max|d|={max_d:.3f}')

print('\n--- Ipotesi verifica ---')
for cl, expected_zone in [(0, 'fronto-centrale (F2/C2/FC2)'), (1, 'F3-PO8')]:
    gamma_diff = RESULTS[cl]['diff']['gamma']
    gamma_sig  = RESULTS[cl]['pval']['gamma'] < 0.05
    n_pos_sig  = (gamma_diff > 0) & gamma_sig  # corretto > sbagliato AND significativo
    print(f'  C{cl} gamma: {n_pos_sig.sum()} elettrodi corretto>sbagliato AND p<.05')
    print(f'     → zona attesa: {expected_zone}')

print('\nFigure salvate:')
for f in sorted(FIG_DIR.glob('eeg22_*.png')):
    print(f'  {f.name}')